# GPT-2 from Scratch: Training Pipeline

**Objective**: Build and train a GPT-2 model using TensorFlow with proper data pipelines, weight loading, and training infrastructure.

## Roadmap
1. **Environment Setup** - Imports, device config, precision settings
2. **Configuration** - Pydantic-based hyperparameter management
3. **Data Pipeline** - TFRecord loading and preprocessing
4. **Architecture** - Clean layer implementations with docstrings
5. **Weight Loading** - Load HuggingFace GPT-2 weights with validation
6. **Training Utilities** - Loss, optimizer, scheduler, checkpointing
7. **Training Loop** - Main training with validation and early stopping
8. **Inference** - Text generation utilities


## 1. Environment Setup & Dependencies


<!-- textbook-note -->
### Textbook Note: Environment Setup

A training notebook is more than a list of imports. It defines the numerical and execution environment in which every later experiment will run. For deep learning, this matters because the same Python code can behave differently depending on TensorFlow version, device placement, mixed precision policy, random seeds, and installed tokenizer/model-loading libraries.

Key definitions:

- **Runtime**: the actual Python + TensorFlow process that executes the graph or eager operations.
- **Device placement**: TensorFlow's decision about whether an operation runs on CPU, GPU, or TPU.
- **Precision policy**: the default floating-point dtype used by layers and variables. In this notebook, precision is intentionally controlled because language models are sensitive to numerical instability.
- **Reproducibility**: the practice of setting seeds and controlling randomness so that debugging is possible.

Why this section matters: if training becomes unstable, you need to know whether the instability comes from the model, data, optimizer, precision policy, or environment. Clean setup reduces the number of unknowns.


In [ ]:
# Install required packages

!pip install -q pydantic pydantic-settings tiktoken transformers huggingface-hub safetensors

In [ ]:
# Core imports
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# Config and utilities
from pydantic import Field, field_validator, ConfigDict
from pydantic_settings import BaseSettings
import tiktoken
from tqdm import tqdm

# For weight loading and HF models
from huggingface_hub import hf_hub_download
from safetensors import safe_open
import torch
from transformers import GPT2LMHeadModel

print("✓ All imports successful")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Set precision policy (float32 for numerical stability during training)
tf.keras.mixed_precision.set_global_policy("float32")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
torch.manual_seed(42)

print("✓ Precision policy set to float32")
print("✓ Random seeds fixed for reproducibility")

## 2. Configuration Management

Using Pydantic for type-safe, validated hyperparameter management.


<!-- textbook-note -->
### Textbook Note: Configuration As The Contract

A configuration object is the contract between the model, data pipeline, optimizer, and training loop. Instead of scattering values like `n_layer`, `block_size`, or `learning_rate` across the notebook, they live in one validated object.

Important GPT-2 configuration terms:

- **`vocab_size`**: number of token IDs the model can represent. GPT-2 uses 50,257 tokens.
- **`block_size`**: maximum context length seen by the model during one forward pass.
- **`n_embedd`**: width of each token representation. This is the channel dimension of the transformer.
- **`n_head`**: number of attention heads. Each head attends through a smaller subspace.
- **`n_layer`**: number of decoder blocks stacked sequentially.
- **`d_ff`**: hidden width inside the feed-forward network, usually around 4x `n_embedd`.
- **`warmup_steps`**: number of early steps where the learning rate ramps up gradually.

Why validation matters: many transformer bugs are shape bugs. For example, `n_embedd` must divide evenly by `n_head`, otherwise attention heads cannot split the hidden dimension cleanly.


In [ ]:
class GPTConfig(BaseSettings):
    """Configuration for GPT-2 model training.
    
    Validates all hyperparameters with sensible defaults.
    Can load from environment variables with GPT_ prefix.
    """
    model_config = ConfigDict(env_prefix="GPT_")

    # ============ Model Architecture ============
    vocab_size: int = Field(default=50257, gt=0, description="GPT-2 vocabulary size")
    block_size: int = Field(default=128, gt=0, description="Max sequence length")
    n_layer: int = Field(default=12, gt=0, description="Number of transformer blocks")
    n_head: int = Field(default=12, gt=0, description="Number of attention heads")
    n_embedd: int = Field(default=768, gt=0, description="Embedding dimension")
    d_ff: int = Field(default=3072, gt=0, description="Feed-forward hidden dimension")
    epsilon: float = Field(default=1e-12, ge=0.0, description="LayerNorm epsilon")
    dropout: float = Field(default=0.1, ge=0.0, le=1.0, description="Dropout rate")

    # ============ Training Hyperparameters ============
    batch_size: int = Field(default=8, gt=0, description="Batch size for training")
    learning_rate: float = Field(default=3e-4, gt=0, description="Initial learning rate")
    weight_decay: float = Field(default=0.1, ge=0, description="L2 regularization")
    grad_clip_norm: float = Field(default=1.0, gt=0, description="Gradient clipping threshold")
    warmup_steps: int = Field(default=2000, ge=0, description="Learning rate warmup steps")
    total_steps: int = Field(default=100000, gt=0, description="Total training steps")
    accum_steps: int = Field(default=4, gt=0, description="Gradient accumulation steps")

    # ============ Data & Paths ============
    train_dir: str = Field(default="GPT/data/tfrecords/train", description="Training TFRecord directory")
    val_dir: str = Field(default="GPT/data/tfrecords/val", description="Validation TFRecord directory")
    checkpoint_dir: str = Field(default="checkpoints", description="Checkpoint save directory")
    log_dir: str = Field(default="logs", description="Tensorboard log directory")

    # ============ Evaluation & Logging ============
    eval_interval: int = Field(default=100, gt=0, description="Eval frequency (steps)")
    save_interval: int = Field(default=1000, gt=0, description="Checkpoint save frequency (steps)")
    log_interval: int = Field(default=10, gt=0, description="Logging frequency (steps)")
    num_workers: int = Field(default=4, gt=0, description="Data pipeline workers")

    # ============ Validation ============
    @field_validator("block_size")
    @classmethod
    def validate_block_size(cls, v):
        if v & (v - 1):  # Check if power of 2
            raise ValueError(f"block_size {v} should be a power of 2")
        return v

    @field_validator("n_head")
    @classmethod
    def validate_heads(cls, v, info):
        n_embedd = info.data.get("n_embedd")
        if n_embedd is not None and n_embedd % v != 0:
            raise ValueError(f"n_embedd ({n_embedd}) must be divisible by n_head ({v})")
        return v

    @field_validator("warmup_steps")
    @classmethod
    def validate_warmup(cls, v, info):
        total_steps = info.data.get("total_steps")
        if total_steps is not None and v >= total_steps:
            raise ValueError("warmup_steps must be smaller than total_steps")
        return v

    @field_validator("train_dir", "val_dir", "checkpoint_dir", "log_dir")
    @classmethod
    def validate_paths(cls, v):
        """Create directories if they don't exist."""
        Path(v).mkdir(parents=True, exist_ok=True)
        return str(v)


# Initialize global config
config = GPTConfig()
print(f"✓ Config loaded")
print(f"  Model: {config.n_layer}L, {config.n_head}H, {config.n_embedd}D")
print(f"  Batch: {config.batch_size}, LR: {config.learning_rate}, Warmup: {config.warmup_steps}")

## 3. Data Pipeline

Load and preprocess TFRecord data for training.


<!-- textbook-note -->
### Textbook Note: Language Model Data Pipeline

A decoder-only language model learns next-token prediction. The data pipeline converts raw text into fixed-length token sequences that the model can consume efficiently.

Pipeline flow:

```text
raw text -> tokenizer -> token ids -> fixed-length windows -> TFRecord shards -> tf.data batches
```

Core definitions:

- **Tokenization**: converting text into integer IDs from a vocabulary.
- **Sequence/window**: a fixed-length slice of token IDs, usually length `block_size + 1` so inputs and labels can be shifted.
- **Input tokens**: all tokens except the final token in a window.
- **Label tokens**: all tokens except the first token in a window.
- **TFRecord**: TensorFlow's binary record format for streaming large datasets efficiently.
- **Sharding**: splitting a dataset into multiple files so loading can be parallelized and resumed more easily.

For causal language modeling, the model sees tokens up to position `t` and predicts token `t+1`. This is why the input and label arrays are offset by one position.

Example:

```text
tokens: [10, 25, 91, 44, 8]
input : [10, 25, 91, 44]
label : [25, 91, 44, 8]
```

The model is not taught a separate label class for the whole sequence. It predicts the next token at every position.


In [ ]:
def parse_tfrecord(example_proto, block_size: int):
    """Parse a single TFRecord example.
    
    Expected format: Serialized tensor of token IDs.
    Pads/truncates to block_size and creates input/target pairs.
    """
    # Parse the serialized tensor
    parsed = tf.io.parse_single_example(
        example_proto,
        features={"tokens": tf.io.FixedLenFeature((), tf.string)}
    )
    tokens = tf.io.decode_compressed(parsed["tokens"], compression_type="GZIP")
    tokens = tf.io.parse_tensor(tokens, out_type=tf.int32)
    
    # Pad or truncate to block_size + 1
    seq_len = tf.shape(tokens)[0]
    if seq_len < block_size + 1:
        tokens = tf.pad(tokens, [[0, block_size + 1 - seq_len]])
    else:
        tokens = tokens[:block_size + 1]
    
    # Split into input and target
    input_ids = tokens[:block_size]
    labels = tokens[1:block_size + 1]
    
    return input_ids, labels


def create_dataset(record_dir: str, batch_size: int, block_size: int, shuffle_buffer: int = 10000):
    """Create a TFRecord dataset pipeline.
    
    Args:
        record_dir: Directory containing .tfrecord files
        batch_size: Batch size for training
        block_size: Max sequence length
        shuffle_buffer: Buffer size for shuffling
    
    Returns:
        Batched tf.data.Dataset
    """
    # List all TFRecord files
    files = tf.data.Dataset.list_files(f"{record_dir}/*.tfrecord")
    
    # Load and parse
    dataset = files.interleave(
        lambda f: tf.data.TFRecordDataset(f),
        cycle_length=4,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    
    # Parse, shuffle, batch
    dataset = dataset.map(
        lambda x: parse_tfrecord(x, block_size),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    dataset = dataset.shuffle(shuffle_buffer)
    dataset = dataset.batch(batch_size, drop_remainder=True)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset


# Create datasets
print("Creating datasets...")
train_ds = create_dataset(config.train_dir, config.batch_size, config.block_size)
val_ds = create_dataset(config.val_dir, config.batch_size, config.block_size)

# Peek at a batch
batch_input, batch_labels = next(iter(train_ds))
print(f"✓ Datasets created")
print(f"  Train batch shape: {batch_input.shape}, {batch_labels.shape}")
print(f"  Val batch shape: {batch_input.shape}, {batch_labels.shape}")

## 4. Architecture: Clean Layer Implementations

All layers with clear docstrings explaining the math.


<!-- textbook-note -->
### Textbook Note: GPT-2 Architecture

GPT-2 is a **decoder-only transformer**. It is called decoder-only because it uses the causal self-attention half of the original transformer architecture, without an encoder stack.

High-level forward pass:

```text
input token ids
-> token embedding + position embedding
-> repeated decoder blocks
-> final layer norm
-> vocabulary logits
```

Important components:

- **Token embedding (`wte`)** maps each token ID to a dense vector.
- **Position embedding (`wpe`)** tells the model where each token appears in the sequence.
- **Causal self-attention** lets each token read only previous tokens and itself.
- **Feed-forward network** transforms each position independently after attention has mixed contextual information.
- **Residual connections** let information and gradients pass through many layers.
- **Layer normalization** stabilizes hidden-state distributions.

GPT-2 uses **pre-norm** blocks in many modern implementations:

```text
x -> LayerNorm -> Attention -> residual add
x -> LayerNorm -> MLP       -> residual add
```

The causal mask is the key difference between GPT-style generation and bidirectional models like BERT. It prevents the model from seeing future tokens during training.


In [ ]:
class GPTEmbeddings(tf.keras.layers.Layer):
    """Token and positional embeddings.
    
    Input: (batch, seq_len) token IDs
    Output: (batch, seq_len, n_embedd) embeddings
    """
    def __init__(self, config):
        super().__init__()
        self.wte = tf.keras.layers.Embedding(config.vocab_size, config.n_embedd)  # Token embeddings
        self.wpe = tf.keras.layers.Embedding(config.block_size, config.n_embedd)   # Position embeddings

    def call(self, input_ids):
        T = tf.shape(input_ids)[1]
        pos = tf.range(T)[tf.newaxis, :]
        tok_emb = self.wte(input_ids)
        pos_emb = self.wpe(pos)
        return tok_emb + pos_emb


class MultiHeadAttention(tf.keras.layers.Layer):
    """Multi-head self-attention with causal masking.
    
    Input: (batch, seq_len, n_embedd)
    Output: (batch, seq_len, n_embedd)
    
    Process:
    1. Project to Q, K, V
    2. Split into multiple heads
    3. Compute attention scores with causal mask
    4. Apply softmax and aggregate
    5. Project output
    """
    def __init__(self, config):
        super().__init__()
        self.h = config.n_head
        self.d_model = config.n_embedd
        self.d_k = config.n_embedd // config.n_head

        assert config.n_embedd % config.n_head == 0, "n_embedd must be divisible by n_head"
        
        self.c_attn = tf.keras.layers.Dense(3 * config.n_embedd)  # Combined QKV projection
        self.c_proj = tf.keras.layers.Dense(config.n_embedd)      # Output projection

    def _create_causal_mask(self, seq_len):
        """Create causal attention mask: 0 where we attend, -1e4 where we don't."""
        lower_tri = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        mask = 1.0 - lower_tri
        return mask[tf.newaxis, tf.newaxis, :, :]

    def call(self, X):
        B = tf.shape(X)[0]
        T = tf.shape(X)[1]
        D = self.d_model
        h = self.h
        dk = self.d_k

        # Project and split into heads
        qkv = self.c_attn(X)
        Q, K, V = tf.split(qkv, 3, axis=-1)
        
        Q = tf.reshape(Q, [B, T, h, dk])
        K = tf.reshape(K, [B, T, h, dk])
        V = tf.reshape(V, [B, T, h, dk])
        
        Q = tf.transpose(Q, [0, 2, 1, 3])  # (B, h, T, dk)
        K = tf.transpose(K, [0, 2, 1, 3])
        V = tf.transpose(V, [0, 2, 1, 3])

        # Compute attention scores
        scores = tf.matmul(Q, tf.transpose(K, [0, 1, 3, 2]))
        scores = scores / tf.sqrt(tf.cast(dk, scores.dtype))

        # Apply causal mask
        mask = self._create_causal_mask(T)
        scores = scores + mask * tf.cast(-1e4, scores.dtype)

        # Softmax and aggregate
        weights = tf.nn.softmax(scores, axis=-1)
        context = tf.matmul(weights, V)  # (B, h, T, dk)

        # Reshape and project
        context = tf.transpose(context, [0, 2, 1, 3])  # (B, T, h, dk)
        context = tf.reshape(context, [B, T, D])

        return self.c_proj(context)


class FFN(tf.keras.layers.Layer):
    """Feed-forward network: Linear -> GELU -> Linear.
    
    Input: (batch, seq_len, n_embedd)
    Output: (batch, seq_len, n_embedd)
    
    Architecture:
    - Expand to 4x dimension
    - Apply GELU activation
    - Project back to original dimension
    """
    def __init__(self, config):
        super().__init__()
        self.c_fc = tf.keras.layers.Dense(config.d_ff)
        self.c_proj = tf.keras.layers.Dense(config.n_embedd)

    def call(self, X):
        X = self.c_fc(X)
        X = tf.nn.gelu(X, approximate=False)  
        X = self.c_proj(X)
        return X


class DecoderBlock(tf.keras.layers.Layer):
    """Transformer decoder block: LayerNorm -> Attention -> Residual -> LayerNorm -> FFN -> Residual.
    
    Pre-norm residual connections for training stability.
    """
    def __init__(self, config):
        super().__init__()
        self.ln_1 = tf.keras.layers.LayerNormalization(epsilon=config.epsilon)
        self.attn = MultiHeadAttention(config)
        self.ln_2 = tf.keras.layers.LayerNormalization(epsilon=config.epsilon)
        self.mlp = FFN(config)

    def call(self, X):
        # Attention with pre-norm residual
        attn_out = self.attn(self.ln_1(X))
        X = X + attn_out
        
        # FFN with pre-norm residual
        mlp_out = self.mlp(self.ln_2(X))
        X = X + mlp_out
        
        return X


class Decoder(tf.keras.Model):
    """Full GPT-2 model: Embeddings -> Transformer blocks -> Output projection.
    
    Input: (batch, seq_len) token IDs
    Output: (batch, seq_len, vocab_size) logits
    """
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        self.embedding = GPTEmbeddings(config)
        self.decoder_layers = [DecoderBlock(config) for _ in range(config.n_layer)]
        self.ln_f = tf.keras.layers.LayerNormalization(epsilon=config.epsilon)

    def call(self, input_ids, return_hidden: bool = False):
        """Forward pass.
        
        Args:
            input_ids: (batch, seq_len) token IDs
            return_hidden: If True, return dict of all intermediate activations
        
        Returns:
            logits: (batch, seq_len, vocab_size)
            or dict of activations if return_hidden=True
        """
        activations = {}

        # Embeddings
        x = self.embedding(input_ids)
        activations['embedding'] = x

        # Transformer blocks
        for i, block in enumerate(self.decoder_layers):
            x = block(x)
            activations[f'block_{i}'] = x

        # Final layer norm
        x = self.ln_f(x)
        activations['ln_f'] = x
        
        # Project to vocab
        logits = tf.matmul(x, self.embedding.wte.embeddings, transpose_b=True)
        activations['logits'] = logits

        if return_hidden:
            return activations
        return logits


print("✓ All architecture classes defined")

## 5. Build Model


<!-- textbook-note -->
### Textbook Note: Building Subclassed Keras Models

In TensorFlow/Keras subclassing, layers are created in `__init__`, but variables are often created only when the model first sees input data. This is why many workflows run a dummy forward pass before loading weights or printing summaries.

The model build step verifies:

- input shape compatibility,
- attention head splitting,
- output logits shape,
- variable creation,
- parameter count.

Expected GPT logits shape:

```text
input_ids: (batch, sequence)
logits   : (batch, sequence, vocab_size)
```

A common mistake is expecting the model to output only one vector per sequence. GPT outputs one vocabulary distribution per token position, because every position is a next-token training example.


In [ ]:
# Create model
print("Building model...")
model = Decoder(config=config)

# Build with dummy input to initialize weights
dummy_input = tf.constant([[15496, 11, 616, 3290]], dtype=tf.int32)
_ = model(dummy_input)

print(f"✓ Model built")
print(f"  Total parameters: {sum([tf.size(w).numpy() for w in model.trainable_variables])/1e6:.2f}M")

## 6. Load HuggingFace Weights

Load pretrained GPT-2 weights from HuggingFace and validate against reference implementation.


<!-- textbook-note -->
### Textbook Note: Pretrained Weight Loading

Loading HuggingFace GPT-2 weights is a compatibility test between your implementation and the reference architecture. If the layer names, shapes, transposes, or activation functions differ, the loaded model may run but produce incorrect logits.

Important ideas:

- **Checkpoint**: saved tensors from a trained model.
- **Tensor mapping**: assigning checkpoint tensors to local model variables.
- **Shape compatibility**: local variables and checkpoint tensors must match, sometimes after transposition.
- **Weight tying**: GPT-2 uses the token embedding matrix as the output projection matrix.

Why this matters: if your randomly initialized model has the same architecture as GPT-2, it should accept pretrained GPT-2 weights. This gives you a strong signal that your architecture is not merely shape-compatible, but semantically close to the original model.

After loading weights, always run a validation prompt and compare logits against a trusted implementation when possible.


In [ ]:
def load_hf_weights(model, hf_weights, config):
    """Load HuggingFace weights into TensorFlow model.
    
    Note: safetensors with framework='tf' automatically handles transposes.
    """
    print("Loading weights...")
    
    # Embeddings (no transpose needed)
    model.embedding.wte.embeddings.assign(hf_weights["wte.weight"])
    model.embedding.wpe.embeddings.assign(hf_weights["wpe.weight"][:config.block_size])

    # Final LayerNorm
    model.ln_f.weights[0].assign(hf_weights["ln_f.weight"])
    model.ln_f.weights[1].assign(hf_weights["ln_f.bias"])

    # Decoder blocks
    for i in range(config.n_layer):
        block = model.decoder_layers[i]

        # LayerNorms (no transpose)
        block.ln_1.weights[0].assign(hf_weights[f'h.{i}.ln_1.weight'])
        block.ln_1.weights[1].assign(hf_weights[f'h.{i}.ln_1.bias'])
        block.ln_2.weights[0].assign(hf_weights[f'h.{i}.ln_2.weight'])
        block.ln_2.weights[1].assign(hf_weights[f'h.{i}.ln_2.bias'])

        # Attention
        block.attn.c_attn.kernel.assign(hf_weights[f"h.{i}.attn.c_attn.weight"])
        block.attn.c_attn.bias.assign(hf_weights[f"h.{i}.attn.c_attn.bias"])
        block.attn.c_proj.kernel.assign(hf_weights[f"h.{i}.attn.c_proj.weight"])
        block.attn.c_proj.bias.assign(hf_weights[f"h.{i}.attn.c_proj.bias"])

        # FFN
        block.mlp.c_fc.kernel.assign(hf_weights[f"h.{i}.mlp.c_fc.weight"])
        block.mlp.c_fc.bias.assign(hf_weights[f"h.{i}.mlp.c_fc.bias"])
        block.mlp.c_proj.kernel.assign(hf_weights[f"h.{i}.mlp.c_proj.weight"])
        block.mlp.c_proj.bias.assign(hf_weights[f"h.{i}.mlp.c_proj.bias"])

        if (i + 1) % 4 == 0:
            print(f"  ✓ Blocks 0-{i} loaded")

    print("✓ All weights loaded!")


# Download and load
print("Downloading HuggingFace GPT-2...")
checkpoint_path = hf_hub_download(repo_id="gpt2", filename="model.safetensors")

hf_weights = {}
with safe_open(checkpoint_path, framework="tf") as f:
    for key in f.keys():
        hf_weights[key] = f.get_tensor(key)

print(f"✓ Downloaded {len(hf_weights)} tensors")
load_hf_weights(model, hf_weights, config)

## 7. Validate Against HuggingFace

Quick sanity check: compare model outputs with reference implementation.


<!-- textbook-note -->
### Textbook Note: Model Validation

Validation here means checking whether your TensorFlow implementation behaves like the reference HuggingFace model after loading the same weights.

What to compare:

- output tensor shape,
- sample logits,
- maximum absolute difference,
- generated next tokens for a simple prompt.

Small differences can happen because of dtype, implementation details, or numerical precision. Large differences usually mean one of these is wrong:

- QKV weight ordering,
- Dense kernel transpose,
- LayerNorm epsilon,
- GELU approximation,
- positional embedding alignment,
- causal mask behavior.

This validation step is a bridge between theory and implementation. It proves that the equations you coded match the architecture you intended.


In [ ]:
print("\n" + "="*70)
print("VALIDATION: TensorFlow vs HuggingFace")
print("="*70)

# Test input
input_ids_tf = tf.constant([[15496, 11, 616, 3290]], dtype=tf.int32)
input_ids_pt = torch.tensor([[15496, 11, 616, 3290]])

# TensorFlow outputs
print("\nRunning TensorFlow model...")
my_outputs = model(input_ids_tf, return_hidden=True)

# HuggingFace outputs
print("Running HuggingFace model...")
hf_model = GPT2LMHeadModel.from_pretrained("gpt2")
hf_model.eval()

with torch.no_grad():
    hf_outputs = hf_model(input_ids_pt, output_hidden_states=True)

hf_logits = tf.convert_to_tensor(hf_outputs.logits.cpu().numpy(), dtype=tf.float32)
hf_hidden = [
    tf.convert_to_tensor(h.cpu().numpy(), dtype=tf.float32)
    for h in hf_outputs.hidden_states
]

# Compare
print("\n" + "-"*70)
print("Hidden State Comparison (Max Absolute Error)")
print("-"*70 + "\n")

diffs = []
emb_diff = tf.reduce_max(tf.abs(my_outputs['embedding'] - hf_hidden[0])).numpy()
print(f"Embedding : {emb_diff:.8f} {'✓' if emb_diff < 0.01 else '~' if emb_diff < 0.1 else '✗'}")

for i in range(12):
    block_diff = tf.reduce_max(tf.abs(my_outputs[f'block_{i}'] - hf_hidden[i+1])).numpy()
    diffs.append(block_diff)
    status = "✓" if block_diff < 0.01 else "~" if block_diff < 0.1 else "✗"
    print(f"Block {i:2d} : {block_diff:.8f} {status}")

ln_diff = tf.reduce_max(tf.abs(my_outputs['ln_f'] - hf_hidden[-1])).numpy()
logits_diff = tf.reduce_max(tf.abs(my_outputs['logits'] - hf_logits)).numpy()

print(f"\nFinal LN  : {ln_diff:.8f}")
print(f"Logits    : {logits_diff:.8f}")

print("\n" + "-"*70)
avg_diff = np.mean(diffs)
max_diff = np.max(diffs)

print(f"Average block difference: {avg_diff:.8f}")
print(f"Max block difference:     {max_diff:.8f}")

if max_diff < 0.01:
    print("\n✓✓✓ PERFECT MATCH! ✓✓✓")
elif max_diff < 0.1:
    print("\n✓✓ GOOD MATCH (acceptable) ✓✓")
else:
    print(f"\n⚠ Large differences - check weight loading")

print("="*70)

## 8. Training Utilities

Loss function, optimizer, learning rate scheduler, and checkpointing.


<!-- textbook-note -->
### Textbook Note: Loss, Optimizer, Schedule, Checkpoints

Training utilities are the support system around the model. They do not define the architecture, but they decide whether the architecture can learn stably.

Core definitions:

- **Cross-entropy loss**: measures how much probability the model assigns to the correct next token.
- **Perplexity**: `exp(loss)`. Lower perplexity means the model is less surprised by the data.
- **Optimizer**: updates weights using gradients. AdamW is common for transformer training.
- **Weight decay**: penalizes large weights to improve generalization.
- **Gradient clipping**: limits update magnitude when gradients spike.
- **Checkpointing**: saves model state so training can resume or inference can use a trained model.

Training stability usually depends more on these utilities than beginners expect. A correct model with a bad learning rate schedule can still fail.


In [ ]:
def create_optimizer(config):
    """Create AdamW optimizer with learning rate schedule.
    
    Uses linear warmup followed by cosine decay.
    """
    lr_schedule = tf.keras.optimizers.schedules.PolynomialDecay(
        initial_learning_rate=config.learning_rate,
        decay_steps=config.total_steps - config.warmup_steps,
        end_learning_rate=0.0,
        power=1.0  # Linear decay
    )
    
    # Apply warmup manually if needed, or use learning rate schedule
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=config.weight_decay,
        clipnorm=config.grad_clip_norm
    )
    
    return optimizer


def compute_loss(logits, labels):
    """Compute cross-entropy loss.
    
    Args:
        logits: (batch, seq_len, vocab_size)
        labels: (batch, seq_len)
    
    Returns:
        Scalar loss
    """
    # Flatten for loss computation
    logits_flat = tf.reshape(logits, [-1, logits.shape[-1]])
    labels_flat = tf.reshape(labels, [-1])
    
    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=labels_flat,
        logits=logits_flat
    )
    
    return tf.reduce_mean(loss)


def compute_perplexity(loss):
    """Compute perplexity from loss."""
    return tf.exp(loss)


class Checkpoint:
    """Simple checkpoint manager."""
    def __init__(self, checkpoint_dir, model, optimizer):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.model = model
        self.optimizer = optimizer
    
    def save(self, step):
        """Save model and optimizer state."""
        ckpt_path = self.checkpoint_dir / f"ckpt_step_{step:06d}"
        self.model.save_weights(str(ckpt_path))
        print(f"✓ Checkpoint saved: {ckpt_path}")
    
    def load(self, step):
        """Load model and optimizer state."""
        ckpt_path = self.checkpoint_dir / f"ckpt_step_{step:06d}"
        self.model.load_weights(str(ckpt_path))
        print(f"✓ Checkpoint loaded: {ckpt_path}")


print("✓ Training utilities defined")

## 9. Training Loop

Main training with validation and checkpointing.


<!-- textbook-note -->
### Textbook Note: The Training Loop

The training loop is where learning actually happens.

One training step:

```text
1. Read a batch.
2. Run forward pass to get logits.
3. Compute next-token loss.
4. Use GradientTape to compute gradients.
5. Clip gradients if needed.
6. Apply optimizer update.
7. Log loss, perplexity, gradient norm, and learning rate.
```

`GradientTape` records operations during the forward pass so TensorFlow can differentiate the loss with respect to trainable variables.

A good training loop separates:

- **train step**: uses gradients and updates weights,
- **validation step**: no gradients, no updates,
- **logging**: records metrics,
- **checkpointing**: saves recoverable state.

This separation is important because bugs often hide when training, validation, logging, and saving are mixed together.


In [ ]:
@tf.function
def train_step(model, batch_inputs, batch_labels, optimizer):
    """Single training step.
    
    Args:
        model: GPT-2 model
        batch_inputs: (batch, seq_len)
        batch_labels: (batch, seq_len)
        optimizer: AdamW optimizer
    
    Returns:
        loss: Scalar loss value
    """
    with tf.GradientTape() as tape:
        logits = model(batch_inputs, training=True)
        loss = compute_loss(logits, batch_labels)
    
    # Compute gradients and update
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    
    return loss


@tf.function
def val_step(model, batch_inputs, batch_labels):
    """Validation step (no gradient computation)."""
    logits = model(batch_inputs, training=False)
    loss = compute_loss(logits, batch_labels)
    return loss


def train_epoch(model, train_ds, val_ds, optimizer, config, checkpoint, epoch):
    """Train for one epoch.
    
    Args:
        model: GPT-2 model
        train_ds: Training dataset
        val_ds: Validation dataset
        optimizer: AdamW optimizer
        config: Configuration
        checkpoint: Checkpoint manager
        epoch: Current epoch number
    
    Returns:
        Dictionary with training metrics
    """
    train_losses = []
    step_counter = 0
    
    print(f"\n{'='*70}")
    print(f"Epoch {epoch}")
    print(f"{'='*70}")
    
    for batch_inputs, batch_labels in tqdm(train_ds, desc="Training"):
        # Training step
        loss = train_step(model, batch_inputs, batch_labels, optimizer)
        train_losses.append(loss.numpy())
        
        step_counter += 1
        
        # Logging
        if step_counter % config.log_interval == 0:
            avg_loss = np.mean(train_losses[-config.log_interval:])
            ppl = compute_perplexity(avg_loss).numpy()
            lr = optimizer.learning_rate.numpy() if hasattr(optimizer.learning_rate, 'numpy') else optimizer.learning_rate
            print(f"[Step {step_counter:05d}] Loss: {avg_loss:.4f}, PPL: {ppl:.2f}, LR: {lr:.2e}")
        
        # Validation
        if step_counter % config.eval_interval == 0:
            val_losses = []
            for val_inputs, val_labels in tqdm(val_ds, desc="Validation"):
                val_loss = val_step(model, val_inputs, val_labels)
                val_losses.append(val_loss.numpy())
            
            val_loss_avg = np.mean(val_losses)
            val_ppl = compute_perplexity(val_loss_avg).numpy()
            print(f"[Validation] Loss: {val_loss_avg:.4f}, PPL: {val_ppl:.2f}")
        
        # Checkpointing
        if step_counter % config.save_interval == 0:
            checkpoint.save(step_counter)
    
    # Epoch summary
    epoch_loss = np.mean(train_losses)
    epoch_ppl = compute_perplexity(epoch_loss).numpy()
    
    print(f"\nEpoch {epoch} Summary")
    print(f"  Avg Loss: {epoch_loss:.4f}")
    print(f"  Perplexity: {epoch_ppl:.2f}")
    print(f"  Total steps: {step_counter}")
    
    return {"loss": epoch_loss, "perplexity": epoch_ppl, "steps": step_counter}


print("✓ Training loop functions defined")

## 10. Run Training

Start training the model.


<!-- textbook-note -->
### Textbook Note: Reading Training Behavior

Running training is an experiment, not just execution. You are watching whether the model, data, and optimizer form a stable learning system.

Healthy signs:

- training loss decreases over time,
- validation loss decreases or tracks training loss,
- perplexity decreases,
- gradient norm stays finite,
- learning rate follows the intended schedule,
- generated samples slowly become more coherent.

Warning signs:

- sudden loss spike,
- NaN or Inf loss,
- validation loss rising while training loss falls,
- gradient norm exploding,
- repeated tokens during generation,
- no improvement after many steps.

When something breaks, debug in this order: data batch -> logits shape -> loss value -> gradient norm -> learning rate -> precision/checkpoint issues.


In [ ]:
# Initialize
optimizer = create_optimizer(config)
checkpoint = Checkpoint(config.checkpoint_dir, model, optimizer)

# Training loop
num_epochs = 1  # Change as needed; data only goes through once if properly shuffled
history = []

try:
    for epoch in range(num_epochs):
        metrics = train_epoch(model, train_ds, val_ds, optimizer, config, checkpoint, epoch + 1)
        history.append(metrics)
except KeyboardInterrupt:
    print("\n⚠ Training interrupted by user")

print("\n✓ Training complete!")

## 11. Text Generation

Inference utilities for generating text.


<!-- textbook-note -->
### Textbook Note: Autoregressive Generation

GPT generates text autoregressively: one token at a time. After each new token is chosen, it is appended to the context and fed back into the model.

Generation loop:

```text
prompt tokens -> logits for next token -> sampling rule -> append token -> repeat
```

Sampling methods:

- **Greedy decoding**: always choose the highest-probability token. Stable but can be repetitive.
- **Temperature**: divides logits before softmax. Lower temperature is more confident; higher temperature is more random.
- **Top-k sampling**: keeps only the `k` most likely tokens before sampling.
- **Top-p sampling**: keeps the smallest set of tokens whose cumulative probability reaches `p`.
- **Beam search**: keeps multiple candidate continuations, useful for some tasks but often less natural for open-ended text.

Training teaches probability distributions. Decoding decides how to use those distributions.


In [ ]:
def generate(model, input_ids, max_new_tokens=50, temperature=1.0, top_k=50):
    """Generate text using the model.
    
    Args:
        model: GPT-2 model
        input_ids: (1, seq_len) starting tokens
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature (higher = more random)
        top_k: Keep only top-k highest probability tokens
    
    Returns:
        Generated token IDs
    """
    generated = input_ids.numpy().tolist()[0]  # Convert to list for easier manipulation
    
    for _ in range(max_new_tokens):
        # Forward pass
        logits = model(tf.constant([generated[-config.block_size:]], dtype=tf.int32), training=False)
        next_logits = logits[0, -1, :] / temperature
        
        # Top-k sampling
        top_k_vals, top_k_indices = tf.nn.top_k(next_logits, k=top_k)
        probs = tf.nn.softmax(top_k_vals)
        next_token = tf.random.categorical(tf.math.log(probs[tf.newaxis, :]), num_samples=1)[0, 0].numpy()
        next_token = top_k_indices[next_token].numpy()
        
        generated.append(next_token)
    
    return tf.constant([generated], dtype=tf.int32)


# Test generation
print("\n" + "="*70)
print("TEXT GENERATION")
print("="*70)

tokenizer = tiktoken.get_encoding("gpt2")
prompt = "The future of"
prompt_ids = tokenizer.encode(prompt)
input_ids = tf.constant([prompt_ids], dtype=tf.int32)

print(f"\nPrompt: {prompt}")
print(f"Prompt IDs: {prompt_ids}")

generated_ids = generate(model, input_ids, max_new_tokens=50, temperature=0.7)
generated_text = tokenizer.decode(generated_ids[0].numpy().tolist())

print(f"\nGenerated text:")
print(generated_text)

## 12. Utilities & Debugging

Additional tools for analysis and debugging.


<!-- textbook-note -->
### Textbook Note: Debugging Language Model Training

LLM debugging is mostly about making invisible quantities visible.

Useful things to inspect:

- batch shapes and token ranges,
- loss before and after a few updates,
- gradient norm,
- learning rate at each step,
- validation perplexity,
- generated samples at fixed prompts,
- checkpoint load/save behavior.

If loss spikes around a later step, likely causes include:

1. learning rate too high after warmup,
2. gradient explosion,
3. corrupt or unusual data batch,
4. mixed precision numerical instability,
5. checkpoint or weight-loading mismatch.

A mature training notebook should help answer: what changed, when did it change, and which signal moved first?


In [ ]:
def plot_training_history(history):
    """Plot training metrics over epochs."""
    if not history:
        print("No history to plot")
        return
    
    losses = [h['loss'] for h in history]
    ppls = [h['perplexity'] for h in history]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].plot(losses)
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True)
    
    axes[1].plot(ppls)
    axes[1].set_title('Perplexity')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('PPL')
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()


def print_model_summary(model, config):
    """Print model architecture summary."""
    print("\n" + "="*70)
    print("MODEL SUMMARY")
    print("="*70)
    
    total_params = 0
    for var in model.trainable_variables:
        num_params = tf.size(var).numpy()
        total_params += num_params
        print(f"{var.name:40s} {var.shape}  {num_params:>12,d}")
    
    print("="*70)
    print(f"Total parameters: {total_params/1e6:.2f}M")
    print(f"Model size (fp32): {total_params * 4 / 1e9:.2f}GB")
    print("="*70)


print("✓ Utility functions defined")